# 01 — Commission decision regex in Court documents

This notebook searches processed InfoCuria Court documents for **references to European Commission decisions**.

The aim is deliberately narrow. It does **not** search for other Court case references, legislation, or general legal citations. It identifies three Commission-related identifier families:

- `decision_case_number` — Commission competition case/procedure numbers such as `IV/61`, `AT.39904`, `COMP/37.792`, `COMP/F-2/32432`, or `COMP/F2/99912`
- `decision_oj_reference` — published Commission decision references such as `98/531/EC` or `2003/706/EC`
- `decision_celex` — explicit CELEX identifiers for Commission decisions

The notebook keeps two complementary views of the regex results. The first contains every individual regex hit and its character position in the clean Court text. The second groups repeated mentions of the same normalized Commission identifier within the same Court document. Preserving the individual hit positions allows the subsequent classification notebook to prepare reproducible text snippets around all mentions of a candidate before classification begins.

## Inputs and outputs

Input: `output/court_infocuria/court_clean_file_manifest.xlsx`, produced by `04_process_court_documents`.

Only rows with `processing_status` equal to `success` or `review` and a usable `clean_file_path` are scanned.

Output:

- `output/court_regex/court_decision_regex.xlsx`
- `logs/court_regex/court_decision_regex_run_info.json`

The Excel workbook contains two views:

- `occurrences` — one row per individual regex hit, including the exact start and end character positions and a short surrounding context;
- `references` — one row per unique normalized Commission reference per Court document, retaining the current grouped view and adding the positions of all underlying hits.

The grouped view is intended as the main input to the later classifier, while the stored hit positions make it possible to construct larger candidate-specific model inputs without rerunning the regex search.

In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import json
import os
import re

import pandas as pd
from tqdm.auto import tqdm

## 1. Paths and settings

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    override = os.environ.get("ECCJEU_PROJECT_ROOT", "").strip()
    if override:
        root = Path(override).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"ECCJEU_PROJECT_ROOT does not exist: {root}")
        return root

    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "code" / "03_matching").exists():
            return candidate

    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the ec_cjeu project root. Run the notebook from within "
        "the repository or set ECCJEU_PROJECT_ROOT."
    )


PROJECT_ROOT = find_project_root()

INPUT_MANIFEST = (
    PROJECT_ROOT
    / "output"
    / "court_infocuria"
    / "court_clean_file_manifest.xlsx"
)

OUTPUT_DIR = PROJECT_ROOT / "output" / "court_regex"
LOG_DIR = PROJECT_ROOT / "logs" / "court_regex"

OUTPUT_XLSX = OUTPUT_DIR / "court_decision_regex.xlsx"
RUN_INFO_JSON = LOG_DIR / "court_decision_regex_run_info.json"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Characters taken on each side of every occurrence when checking
# whether the words "Commission" and/or "Decision" are nearby.
CONTEXT_CHARS = 250


def repo_path(path: Path | str) -> str:
    path = Path(path).resolve()
    try:
        return path.relative_to(PROJECT_ROOT.resolve()).as_posix()
    except Exception:
        return path.as_posix()


def resolve_repo_path(value: Any) -> Path | None:
    text = str(value or "").strip()
    if not text:
        return None

    path = Path(text)

    if path.is_absolute():
        if path.exists():
            return path.resolve()

        # Tolerate old absolute paths containing the repository name.
        parts = list(path.parts)
        for repo_name in ("ec_cjeu", "eccjeu"):
            if repo_name in parts:
                idx = parts.index(repo_name)
                rel = Path(*parts[idx + 1:])
                candidate = PROJECT_ROOT / rel
                if candidate.exists():
                    return candidate.resolve()

        return path.resolve()

    return (PROJECT_ROOT / path).resolve()


print("Project root:", PROJECT_ROOT)
print("Input:", INPUT_MANIFEST)
print("Output:", OUTPUT_XLSX)
print("Context window:", f"±{CONTEXT_CHARS} characters")

Project root: /home/edik/projects/ec_cjeu
Input: /home/edik/projects/ec_cjeu/output/court_infocuria/court_clean_file_manifest.xlsx
Output: /home/edik/projects/ec_cjeu/output/court_regex/court_decision_regex.xlsx
Context window: ±250 characters


## 2. Load the processed Court-document manifest

Notebook 04 has already linked Court metadata to the clean UTF-8 text files. This notebook therefore starts from the clean-file manifest rather than from the raw download manifest.

In [3]:
if not INPUT_MANIFEST.exists():
    raise FileNotFoundError(f"Required clean-file manifest is missing: {INPUT_MANIFEST}")

manifest = pd.read_excel(INPUT_MANIFEST, dtype=str, keep_default_na=False)
manifest.columns = [str(c).strip() for c in manifest.columns]

required_columns = {
    "record_id",
    "affId",
    "procedureId",
    "document_type",
    "clean_file_path",
    "processing_status",
}

missing = sorted(required_columns - set(manifest.columns))
if missing:
    raise ValueError(f"Clean-file manifest is missing required columns: {missing}")

work = manifest.loc[
    manifest["processing_status"].str.strip().str.lower().isin({"success", "review"})
    & manifest["clean_file_path"].str.strip().ne("")
].copy()

work["resolved_text_path"] = work["clean_file_path"].map(resolve_repo_path)
work["text_file_exists"] = work["resolved_text_path"].map(
    lambda p: bool(p is not None and p.exists() and p.is_file())
)

missing_text = work.loc[~work["text_file_exists"]].copy()
work = work.loc[work["text_file_exists"]].reset_index(drop=True)

print(f"Rows in clean-file manifest: {len(manifest):,}")
print(f"Usable processed Court documents: {len(work):,}")
print(f"Missing referenced text files: {len(missing_text):,}")

display(
    work[
        [
            "record_id",
            "affId",
            "procedureId",
            "document_type",
            "processing_status",
            "clean_file_path",
        ]
    ].head(20)
)

Rows in clean-file manifest: 42,847
Usable processed Court documents: 42,847
Missing referenced text files: 0


,record_id,affId,procedureId,document_type,processing_status,clean_file_path
0,id_85906__000001,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET_SOM,success,data/processed/court_infocuria/text/pdf_85906-...
1,id_46950__000002,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET,success,data/processed/court_infocuria/text/html_46950...
2,id_49061__000003,C/0001/02/00000000RP/01,C/0001/02/00000000RP/01/P/01,ARRET,success,data/processed/court_infocuria/text/html_49061...
3,id_55247__000004,C/0001/02/00000000RP/01,C/0001/02/00000000RP/01/P/01,ARRET_SOM,success,data/processed/court_infocuria/text/html_55247...
4,id_65336__000005,C/0001/03/00000000RP/01,C/0001/03/00000000RP/01/P/01,ARRET_SOM,success,data/processed/court_infocuria/text/html_65336...
5,id_49453__000006,C/0001/03/00000000RP/01,C/0001/03/00000000RP/01/P/01,ARRET,success,data/processed/court_infocuria/text/html_49453...
6,id_57812__000007,C/0001/04/00000000RP/01,C/0001/04/00000000RP/01/P/01,ARRET,success,data/processed/court_infocuria/text/html_57812...
7,id_64087__000008,C/0001/04/00000000RP/01,C/0001/04/00000000RP/01/P/01,ARRET_SOM,success,data/processed/court_infocuria/text/html_64087...
8,id_61266__000009,C/0001/05/00000000RP/01,C/0001/05/00000000RP/01/P/01,ARRET_SOM,success,data/processed/court_infocuria/text/html_61266...
9,id_64718__000010,C/0001/05/00000000RP/01,C/0001/05/00000000RP/01/P/01,ARRET,success,data/processed/court_infocuria/text/html_64718...


## 3. Commission-reference regexes

The patterns are intentionally restricted to Commission-related identifiers.

### Commission competition case numbers

Simple forms include `IV/61`, `IV/26.760`, `COMP/37.792`, and `AT.39904`. At least two digits are required, so a string such as `IV/4` is not accepted.

A second pattern captures COMP sub-series such as `COMP/F-2/32432` and `COMP/F2/99912`.

Whitespace is allowed around separators, including line breaks introduced by PDF extraction. For dotted Commission case numbers, whitespace is also allowed around the internal dot so that forms such as `COMP/37 . 792` or a line break around the dot can still be recovered. The normalized output removes this whitespace.

### Published Commission decision references

Only references with an explicit suffix are retained, for example `98/531/EC`, `2003/706/EC`, or `130/63/EEC`. Whitespace and line breaks around the slashes are accepted.

### CELEX identifiers

Explicit Commission-decision CELEX identifiers are retained separately.

In [4]:
# Simple Commission case numbers:
# IV/61, IV/26.760, COMP/37.792, AT.39904, etc.
# Whitespace/newlines are tolerated around the outer separator and around
# dots inside dotted case numbers. A single layout/OCR whitespace break inside
# an undotted number is also tolerated. Normalization removes that whitespace.
COMMISSION_CASE_SIMPLE_RE = re.compile(
    r"""
    (?<![A-Za-z0-9])
    (?P<prefix>COMP|IV|AT)
    \s*
    [./\-]
    \s*
    (?P<number>
        \d{1,2}\s*\.\s*\d{3,5}(?:\s*\.\s*\d+)?
        |
        \d{1,2}\s+\d{3,5}
        |
        \d{2,6}
    )
    (?![A-Za-z0-9.])
    """,
    re.IGNORECASE | re.VERBOSE,
)

# COMP sub-series:
# COMP/F-2/32432, COMP/F2/99912
COMMISSION_CASE_SUBSERIES_RE = re.compile(
    r"""
    (?<![A-Za-z0-9])
    COMP
    \s*/\s*
    (?P<section>[A-Z](?:\s*-\s*)?\d?)
    \s*/\s*
    (?P<number>\d{2,6})
    (?![A-Za-z0-9])
    """,
    re.IGNORECASE | re.VERBOSE,
)

# Published Commission decision / OJ references.
# Only suffix-bearing forms are retained. Whitespace/newlines around
# separators are tolerated and removed during normalization.
DECISION_OJ_REFERENCE_RE = re.compile(
    r"""
    (?<![A-Za-z0-9])
    (?P<part1>\d{1,4})
    \s*/\s*
    (?P<part2>\d{1,4})
    \s*/\s*
    (?P<suffix>EEC|EC|EU|ECSC|EURATOM)
    (?![A-Za-z0-9])
    """,
    re.IGNORECASE | re.VERBOSE,
)

# Explicit CELEX decision identifiers only.
CELEX_RE = re.compile(
    r"""
    (?<![A-Za-z0-9])
    3
    (?P<year>\d{4})
    D
    (?P<number>\d{4})
    (?![A-Za-z0-9])
    """,
    re.IGNORECASE | re.VERBOSE,
)

COMMISSION_WORD_RE = re.compile(r"\bCommission\b", re.IGNORECASE)
DECISION_WORD_RE = re.compile(r"\bDecision\b", re.IGNORECASE)


def normalize_case_simple(match: re.Match) -> str:
    prefix = match.group("prefix").upper()
    number = re.sub(r"\s+", "", match.group("number"))

    if prefix == "AT":
        return f"AT.{number}"
    return f"{prefix}/{number}"


def normalize_case_subseries(match: re.Match) -> str:
    section = re.sub(r"\s+", "", match.group("section")).upper()
    number = re.sub(r"\s+", "", match.group("number"))
    return f"COMP/{section}/{number}"


def normalize_oj_reference(match: re.Match) -> str:
    return (
        f"{match.group('part1')}/"
        f"{match.group('part2')}/"
        f"{match.group('suffix').upper()}"
    )


def normalize_celex(match: re.Match) -> str:
    return f"3{match.group('year')}D{match.group('number')}"


REGEX_FAMILIES = [
    (
        "decision_case_number",
        COMMISSION_CASE_SUBSERIES_RE,
        normalize_case_subseries,
    ),
    (
        "decision_case_number",
        COMMISSION_CASE_SIMPLE_RE,
        normalize_case_simple,
    ),
    (
        "decision_oj_reference",
        DECISION_OJ_REFERENCE_RE,
        normalize_oj_reference,
    ),
    (
        "decision_celex",
        CELEX_RE,
        normalize_celex,
    ),
]

## 4. Scan the Court texts and build two views

Each regex occurrence is retained with its exact character position in the clean Court text. A short context window is also stored to make the raw hit easy to review.

The same occurrences are then grouped by Court document and normalized Commission identifier. Repeated mentions therefore remain visible at occurrence level while the grouped table provides one candidate for later classification.

For each grouped candidate, `hit_spans_json` stores every `[start_char, end_char]` pair. The subsequent classification notebook can use these positions to prepare and save the exact model input — for example the beginning and end of a long judgment together with larger windows around every mention — before any classification requests are sent.

In [5]:
def collapse_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "")).strip()


def occurrence_context(text: str, start: int, end: int) -> str:
    left = max(0, start - CONTEXT_CHARS)
    right = min(len(text), end + CONTEXT_CHARS)
    return text[left:right]


def scan_text(text: str) -> list[dict[str, Any]]:
    occurrences: list[dict[str, Any]] = []

    for reference_type, pattern, normalizer in REGEX_FAMILIES:
        for match in pattern.finditer(text):
            context = occurrence_context(text, match.start(), match.end())

            commission_nearby = bool(COMMISSION_WORD_RE.search(context))
            decision_nearby = bool(DECISION_WORD_RE.search(context))

            occurrences.append(
                {
                    "reference_type": reference_type,
                    "matched_text": match.group(0),
                    "reference_normalized": normalizer(match),
                    "start_char": int(match.start()),
                    "end_char": int(match.end()),
                    "commission_nearby": commission_nearby,
                    "decision_nearby": decision_nearby,
                    "commission_and_decision_nearby": (
                        commission_nearby and decision_nearby
                    ),
                    "context": collapse_whitespace(context),
                }
            )

    return occurrences


# Cache by physical clean-text path so duplicate manifest rows do not cause the
# same text file to be read and scanned repeatedly.
scan_cache: dict[str, list[dict[str, Any]]] = {}

occurrence_rows: list[dict[str, Any]] = []
grouped_rows: list[dict[str, Any]] = []

for _, doc in tqdm(
    work.iterrows(),
    total=len(work),
    desc="Scanning Court documents for Commission references",
):
    text_path = Path(doc["resolved_text_path"])
    cache_key = str(text_path.resolve())

    if cache_key not in scan_cache:
        text = text_path.read_text(encoding="utf-8", errors="replace")
        scan_cache[cache_key] = scan_text(text)

    occurrences = scan_cache[cache_key]
    if not occurrences:
        continue

    # Occurrence-level view: one row for every individual regex hit.
    for occurrence_index, occ in enumerate(occurrences, start=1):
        occurrence_rows.append(
            {
                "record_id": doc.get("record_id", ""),
                "affId": doc.get("affId", ""),
                "procedureId": doc.get("procedureId", ""),
                "document_type": doc.get("document_type", ""),
                "document_date": doc.get("document_date", ""),
                "case_number_clean": doc.get("case_number_clean", ""),
                "clean_file_path": doc.get("clean_file_path", ""),
                "occurrence_index": occurrence_index,
                **occ,
            }
        )

    # Grouped view: one row per document × normalized regex candidate.
    occ_df = pd.DataFrame(occurrences)

    for (reference_type, reference_normalized), group in occ_df.groupby(
        ["reference_type", "reference_normalized"],
        sort=False,
        dropna=False,
    ):
        group = group.sort_values(["start_char", "end_char"])
        first = group.iloc[0]

        matched_forms = list(dict.fromkeys(group["matched_text"].astype(str)))
        hit_spans = [
            [int(start), int(end)]
            for start, end in zip(group["start_char"], group["end_char"])
        ]

        grouped_rows.append(
            {
                "record_id": doc.get("record_id", ""),
                "affId": doc.get("affId", ""),
                "procedureId": doc.get("procedureId", ""),
                "document_type": doc.get("document_type", ""),
                "document_date": doc.get("document_date", ""),
                "case_number_clean": doc.get("case_number_clean", ""),
                "clean_file_path": doc.get("clean_file_path", ""),
                "reference_type": reference_type,
                "reference": " | ".join(matched_forms),
                "reference_normalized": reference_normalized,
                "reference_count": int(len(group)),
                "hit_spans_json": json.dumps(hit_spans, ensure_ascii=False),
                "commission_nearby_count": int(group["commission_nearby"].sum()),
                "decision_nearby_count": int(group["decision_nearby"].sum()),
                "commission_and_decision_nearby_count": int(
                    group["commission_and_decision_nearby"].sum()
                ),
                "context": first["context"],
            }
        )


OCCURRENCE_COLUMNS = [
    "record_id",
    "affId",
    "procedureId",
    "document_type",
    "document_date",
    "case_number_clean",
    "clean_file_path",
    "occurrence_index",
    "reference_type",
    "matched_text",
    "reference_normalized",
    "start_char",
    "end_char",
    "commission_nearby",
    "decision_nearby",
    "commission_and_decision_nearby",
    "context",
]

GROUPED_COLUMNS = [
    "record_id",
    "affId",
    "procedureId",
    "document_type",
    "document_date",
    "case_number_clean",
    "clean_file_path",
    "reference_type",
    "reference",
    "reference_normalized",
    "reference_count",
    "hit_spans_json",
    "commission_nearby_count",
    "decision_nearby_count",
    "commission_and_decision_nearby_count",
    "context",
]

occurrence_results = pd.DataFrame(occurrence_rows, columns=OCCURRENCE_COLUMNS)
results = pd.DataFrame(grouped_rows, columns=GROUPED_COLUMNS)

if len(occurrence_results):
    occurrence_results = (
        occurrence_results.sort_values(
            ["affId", "procedureId", "record_id", "start_char", "end_char"],
            kind="stable",
        )
        .reset_index(drop=True)
    )

if len(results):
    results = (
        results.sort_values(
            [
                "affId",
                "procedureId",
                "record_id",
                "reference_type",
                "reference_normalized",
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )

print(f"Documents scanned: {len(work):,}")
print(
    "Documents with at least one Commission-reference candidate:",
    f"{results['record_id'].nunique() if len(results) else 0:,}",
)
print(f"Individual regex hits: {len(occurrence_results):,}")
print(f"Unique document-reference rows: {len(results):,}")

if len(results):
    display(
        results["reference_type"]
        .value_counts()
        .rename_axis("reference_type")
        .reset_index(name="rows")
    )

display(results.head(30))

Scanning Court documents for Commission references:   0%|          | 0/42847 [00:00<?, ?it/s]

Documents scanned: 42,847
Documents with at least one Commission-reference candidate: 16,494
Individual regex hits: 72,883
Unique document-reference rows: 39,291


,reference_type,rows
0,decision_oj_reference,37058
1,decision_case_number,2233


,record_id,affId,procedureId,document_type,document_date,case_number_clean,clean_file_path,reference_type,reference,reference_normalized,reference_count,hit_spans_json,commission_nearby_count,decision_nearby_count,commission_and_decision_nearby_count,context
0,id_46950__000002,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET,2001-12-13,C-1/00,data/processed/court_infocuria/text/html_46950...,decision_oj_reference,1999/514/EC,1999/514/EC,3,"[[1277, 1288], [3405, 3416], [70381, 70392]]",3,3,3,Decision 94/474/EC and repealing Decision 96/2...
1,id_46950__000002,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET,2001-12-13,C-1/00,data/processed/court_infocuria/text/html_46950...,decision_oj_reference,64/433/EEC,64/433/EEC,1,"[[19302, 19312]]",1,1,1,"t of the experience gained, the Commission wil..."
2,id_46950__000002,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET,2001-12-13,C-1/00,data/processed/court_infocuria/text/html_46950...,decision_oj_reference,77/99/EEC,77/99/EEC,1,"[[19344, 19353]]",1,1,1,will endeavour to clarify and complete if need...
3,id_46950__000002,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET,2001-12-13,C-1/00,data/processed/court_infocuria/text/html_46950...,decision_oj_reference,89/662/EEC,89/662/EEC,4,"[[5533, 5543], [5712, 5722], [10476, 10486], [...",2,1,1,7 December 1992 laying down animal health and ...
4,id_46950__000002,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET,2001-12-13,C-1/00,data/processed/court_infocuria/text/html_46950...,decision_oj_reference,90/425/EEC,90/425/EEC,1,"[[5008, 5018]]",0,1,0,porting from its territory to the other Member...
5,id_46950__000002,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET,2001-12-13,C-1/00,data/processed/court_infocuria/text/html_46950...,decision_oj_reference,92/118/EEC,92/118/EEC,1,"[[5268, 5278]]",0,0,0,of 26 June 1990 concerning veterinary and zoot...
6,id_46950__000002,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET,2001-12-13,C-1/00,data/processed/court_infocuria/text/html_46950...,decision_oj_reference,94/474/EC,94/474/EC,3,"[[1036, 1045], [3122, 3131], [70186, 70195]]",3,3,3,"PLICATION for a declaration that, by refusing ..."
7,id_46950__000002,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET,2001-12-13,C-1/00,data/processed/court_infocuria/text/html_46950...,decision_oj_reference,96/239/EC,96/239/EC,4,"[[1069, 1078], [3155, 3164], [4564, 4573], [70...",4,4,4,by refusing to adopt the measures necessary in...
8,id_46950__000002,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET,2001-12-13,C-1/00,data/processed/court_infocuria/text/html_46950...,decision_oj_reference,97/404/EC,97/404/EC,1,"[[12279, 12288]]",1,1,1,orwarded to the Commission the opinion issued ...
9,id_46950__000002,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET,2001-12-13,C-1/00,data/processed/court_infocuria/text/html_46950...,decision_oj_reference,98/256/EC,98/256/EC,4,"[[908, 917], [2994, 3003], [18282, 18291], [70...",4,4,4,"lle, and then by R. Loosli-Surrans and G. de B..."


## 5. Save the occurrence-level and grouped reference tables

Before writing the workbook, text fields are cleaned of the small set of control characters that Excel cannot store. This does not otherwise alter the extracted Court text or the recorded character positions.

In [6]:
# Excel's XML format rejects a small set of ASCII control characters that can
# appear in scraped/OCR'd Court text. Sanitize only the export copies.
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE


def sanitize_excel_text(df: pd.DataFrame) -> pd.DataFrame:
    clean = df.copy()
    for col in clean.select_dtypes(include=["object"]).columns:
        clean[col] = clean[col].map(
            lambda value: ILLEGAL_CHARACTERS_RE.sub("", value)
            if isinstance(value, str)
            else value
        )
    return clean


occurrences_excel = sanitize_excel_text(occurrence_results)
results_excel = sanitize_excel_text(results)

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    occurrences_excel.to_excel(writer, sheet_name="occurrences", index=False)
    results_excel.to_excel(writer, sheet_name="references", index=False)

    for sheet_name in ("occurrences", "references"):
        ws = writer.sheets[sheet_name]
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions

    # Keep both tables readable without elaborate formatting.
    occurrence_widths = {
        "A": 22, "B": 18, "C": 18, "D": 16, "E": 14, "F": 18,
        "G": 48, "H": 16, "I": 26, "J": 30, "K": 28, "L": 14,
        "M": 14, "N": 18, "O": 18, "P": 32, "Q": 90,
    }
    for column, width in occurrence_widths.items():
        writer.sheets["occurrences"].column_dimensions[column].width = width

    reference_widths = {
        "A": 22, "B": 18, "C": 18, "D": 16, "E": 14, "F": 18,
        "G": 48, "H": 26, "I": 28, "J": 28, "K": 16, "L": 55,
        "M": 24, "N": 22, "O": 36, "P": 90,
    }
    for column, width in reference_widths.items():
        writer.sheets["references"].column_dimensions[column].width = width


run_info = {
    "created_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    "input_manifest": repo_path(INPUT_MANIFEST),
    "output_file": repo_path(OUTPUT_XLSX),
    "output_sheets": ["occurrences", "references"],
    "context_chars_each_side": CONTEXT_CHARS,
    "eligible_manifest_rows": int(len(work)),
    "missing_clean_text_files": int(len(missing_text)),
    "documents_with_candidates": int(
        results["record_id"].nunique() if len(results) else 0
    ),
    "individual_regex_hits": int(len(occurrence_results)),
    "unique_document_reference_rows": int(len(results)),
}

RUN_INFO_JSON.write_text(
    json.dumps(run_info, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Saved:", repo_path(OUTPUT_XLSX))
print("Sheets: occurrences, references")
print("Run log:", repo_path(RUN_INFO_JSON))

Saved: output/court_regex/court_decision_regex.xlsx
Sheets: occurrences, references
Run log: logs/court_regex/court_decision_regex_run_info.json
